In [1]:
import logging
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# logging.basicConfig(level=logging.DEBUG)
n_epoch = 1000
# n_epoch = 10

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.LBGFSLS(model=model, line_search_cond="wolfe", line_search_method="interpolate")
# opt = torch_numopt.LBGFSLS(model=model)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.18959656357765198
epoch:  1, loss: 0.03793440759181976
epoch:  2, loss: 0.03530079498887062
epoch:  3, loss: 0.035284098237752914
epoch:  4, loss: 0.035284094512462616
epoch:  5, loss: 0.03528407961130142
epoch:  6, loss: 0.035284072160720825
epoch:  7, loss: 0.03528405725955963
epoch:  8, loss: 0.035284046083688736
epoch:  9, loss: 0.03528403490781784
epoch:  10, loss: 0.035284023731946945
epoch:  11, loss: 0.03528401255607605
epoch:  12, loss: 0.035284001380205154
epoch:  13, loss: 0.03528398647904396
epoch:  14, loss: 0.035283979028463364
epoch:  15, loss: 0.03528396412730217
epoch:  16, loss: 0.035283952951431274
epoch:  17, loss: 0.03528394177556038
epoch:  18, loss: 0.035283930599689484
epoch:  19, loss: 0.03528391942381859
epoch:  20, loss: 0.035283904522657394
epoch:  21, loss: 0.0352838933467865
epoch:  22, loss: 0.035283878445625305
epoch:  23, loss: 0.03528386726975441
epoch:  24, loss: 0.03528385981917381
epoch:  25, loss: 0.03528384864330292
epoch:  26, 

In [6]:
model.eval()
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.952251144507489
Test metrics:  R2 = 0.9402744282817669
